187. Compare three candidate problems: high-activity risk, anomalous activity, and activity drop

### ML Problem Selection — High-Activity Risk

We selected **high-activity risk** as the primary ML problem.

The available data contains grid-level communication activity measured at hourly intervals, including `total_activity`, SMS activity, call activity, internet activity, and engineered features such as average activity, activity growth, peak ratio, variability, and internet share. This makes it possible to learn patterns in historical activity and use them to predict whether activity is likely to be unusually high in the next time window.

We selected this problem over anomalous activity and activity drop because high-activity risk provides a clear operational use case: identifying grid/time periods that may require investigation before the activity occurs.

The model will predict **future high activity**, not actual network congestion. We do not have network capacity, throughput, latency, packet-loss, or radio-utilization data, so the model cannot legitimately claim that a network is congested.

A positive prediction will therefore be treated as a **risk signal for investigation**, rather than a conclusion that congestion exists.

The prediction will use features calculated from the **trailing window ending at time t**, while the target will describe activity in the **future interval t+1**. This separation ensures that the model is making a genuine prediction rather than simply restating a threshold calculated from the same data.

The high-activity label will be treated as a **proxy training label** based on a documented activity threshold. It represents unusually high observed activity, not confirmed network congestion.

**Business action:** If the model identifies high-activity risk for a grid in the next time window, the operational response is **“investigate”** the grid and, where available, verify the condition using additional network information.


188. Select one primary problem for the model.


### Primary ML Problem

The selected primary problem is **High-Activity Risk**.

The model will predict whether a given grid is likely to experience **high communication activity during the next hourly interval**, using only information available up to the current time.

The prediction is intended to identify grid/time periods that may require operational investigation.

The model does **not** predict or confirm network congestion. The available dataset does not contain capacity, throughput, latency, packet-loss, or radio-utilization measurements required to make such a claim.

Therefore, a positive prediction represents **high-activity risk and an instruction to investigate**, rather than a conclusion that the network is congested.


189. Define the prediction unit: a grid plus a time window.


The prediction unit is one grid for one hourly time window.

For each grid, the model uses information available up to time t to predict whether the grid will have high activity during the next hourly interval t+1

190. Define the target as a future window. The label describes the interval at t+1; the features describe
the trailing window ending at t. See the trap below — this is the difference between a real prediction
problem and a restatement of a threshold

The model will use features calculated from the trailing window ending at time t to predict whether the grid will have high activity during the next hourly interval (t+1).

Features: Information available up to t.
Target: High activity during t+1.

This t/t+1 boundary ensures that future information is not used when making the prediction and prevents data leakage.

191.Define the target strategy precisely. If synthetic threshold labels are used, document them as training
proxies and state the threshold.


The target will be a binary label representing whether the grid has high activity in the next hourly interval (t+1).

1 = High activity
0 = Not high activity

The high-activity label will be created using a documented activity threshold. This threshold is a training proxy for high activity and does not represent confirmed network congestion.

###— Target Strategy

We use the **90th-percentile `total_activity` threshold of 1154.5988**.

* `total_activity > 1154.5988` → High activity (1)
* `total_activity ≤ 1154.5988` → Normal activity (0)

This is a training proxy for unusually high activity and does not represent confirmed network congestion.


192. Define the business action: “investigate
”
, never
“the network is congested”.

A positive prediction means “investigate” the grid and time period.

The model must not claim that the network is congested. The prediction is only a signal that the grid may experience unusually high activity and requires further investigation.

193. Identify the data leakage risks, and state how the t / t+1 boundary prevents each one

The main leakage risk is using information from the future interval t+1 when creating features for the prediction.

To prevent this, all features will be calculated using data available up to time t, while the target will describe activity during t+1.

We will also avoid calculating features and the target from the same time window, which could make the model simply reproduce the target threshold instead of predicting the future.